In [ ]:
# Assignment 1 — Was Any Month of 2025 Unusual?

In [50]:
import os
import pandas as pd
 
def find_repo_root(marker_folder="data", start=None):
    # Search upward until we find the folder that contains data/ --
    # that only exists at the repo root, so this works from any folder.
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(current, marker_folder)):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise FileNotFoundError("No data folder found above here.")
        current = parent
 
RAW = os.path.join(find_repo_root(), "data", "raw")
TEXT_COLUMNS = {"safety_report_id": str, "serious": str,
                "patient_age_unit": str, "country": str, "event_date": str}
df = pd.read_csv(os.path.join(RAW, "adverse_events_10k.csv"), dtype=TEXT_COLUMNS)
df["is_serious"] = df["serious"].map({"1": 1.0, "2": 0.0})
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 70)
 
dates = pd.to_datetime(df["event_date"], format="%Y%m%d")
df["month"] = dates.dt.to_period("M").astype(str)
in_2025 = df[dates.dt.year == 2025]
 
monthly = in_2025.groupby("month").agg(
    reports=("safety_report_id", "count"),
    recorded=("is_serious", "count"),
    serious=("is_serious", "sum"),
    serious_share=("is_serious", "mean"),
)
monthly["serious"] = monthly["serious"].astype(int)
print(monthly.round(3))
print()
print("months:", len(monthly), "| reports:", monthly["reports"].sum(),
      "| rows in 2025:", len(in_2025))
print("highest share:", monthly["serious_share"].idxmax(),
      round(monthly["serious_share"].max(), 3))
print("lowest share: ", monthly["serious_share"].idxmin(),
      round(monthly["serious_share"].min(), 3))
print("2025 overall: ", round(in_2025["is_serious"].mean(), 3))


         reports  recorded  serious  serious_share
month                                             
2025-01      249       249      116          0.466
2025-02      271       271       67          0.247
2025-03      277       277      137          0.495
2025-04      261       261      161          0.617
2025-05      256       256      161          0.629
2025-06      257       257      112          0.436
2025-07      262       262      125          0.477
2025-08      222       222      124          0.559
2025-09      246       246      102          0.415
2025-10      277       274       94          0.343
2025-11      251       250      130          0.520
2025-12      241       239      128          0.536

months: 12 | reports: 3070 | rows in 2025: 3070
highest share: 2025-05 0.629
lowest share:  2025-02 0.247
2025 overall:  0.476


In [ ]:
#  Assignment 2 — Reports by Business Region

In [51]:
import os
import pandas as pd
 
def find_repo_root(marker_folder="data", start=None):
    # Search upward until we find the folder that contains data/ --
    # that only exists at the repo root, so this works from any folder.
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(current, marker_folder)):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise FileNotFoundError("No data folder found above here.")
        current = parent
 
RAW = os.path.join(find_repo_root(), "data", "raw")
TEXT_COLUMNS = {"safety_report_id": str, "serious": str,
                "patient_age_unit": str, "country": str, "event_date": str}
df = pd.read_csv(os.path.join(RAW, "adverse_events_10k.csv"), dtype=TEXT_COLUMNS)
df["is_serious"] = df["serious"].map({"1": 1.0, "2": 0.0})
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 70)
 
codes = pd.read_csv(os.path.join(RAW, "country_codes.csv"),
                    keep_default_na=False, na_values=[""])
print("lookup rows:", len(codes), "| one row per code:", codes["country_code"].is_unique)
 
regions = df.merge(codes, left_on="country", right_on="country_code",
                   how="left", validate="many_to_one", indicator=True)
print("rows before:", len(df), "| after:", len(regions))
print(regions["_merge"].value_counts())
 
has_country = regions["country"].notna()
unmatched = regions[has_country & (regions["_merge"] == "left_only")]
print()
print("reports with a country code the lookup does not know:", len(unmatched))
print("distinct unknown codes:", unmatched["country"].nunique())
print(unmatched["country"].value_counts().head(6))
 
regions["business_region"] = regions["business_region"].fillna("Code not in lookup")
regions.loc[~has_country, "business_region"] = "Not reported"
summary = regions.groupby("business_region").agg(
    reports=("safety_report_id", "count"),
    serious_share=("is_serious", "mean"),
).sort_values("reports", ascending=False)
summary["share_of_reports"] = (summary["reports"] / len(regions)).round(3)
print()
print(summary.round(3))
print("adds up to:", summary["reports"].sum())

lookup rows: 54 | one row per code: True
rows before: 6218 | after: 6218
_merge
both          5494
left_only      724
right_only       0
Name: count, dtype: int64

reports with a country code the lookup does not know: 321
distinct unknown codes: 34
country
EU    229
TW     16
PR     12
PT     12
SA      7
SG      6
Name: count, dtype: int64

                    reports  serious_share  share_of_reports
business_region                                             
North America          4404          0.318             0.708
EMEA                    522          0.943             0.084
APAC                    437          0.899             0.070
Not reported            403          0.568             0.065
Code not in lookup      321          0.916             0.052
LATAM                   131          0.847             0.021
adds up to: 6218


In [ ]:
# Assignment 3 — Which Drug Classes Gained Share Over a Year?

In [52]:
import os
import pandas as pd
 
def find_repo_root(marker_folder="data", start=None):
    # Search upward until we find the folder that contains data/ --
    # that only exists at the repo root, so this works from any folder.
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(current, marker_folder)):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise FileNotFoundError("No data folder found above here.")
        current = parent
 
RAW = os.path.join(find_repo_root(), "data", "raw")
TEXT_COLUMNS = {"safety_report_id": str, "serious": str,
                "patient_age_unit": str, "country": str, "event_date": str}
df = pd.read_csv(os.path.join(RAW, "adverse_events_10k.csv"), dtype=TEXT_COLUMNS)
df["is_serious"] = df["serious"].map({"1": 1.0, "2": 0.0})
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 70)
 
drug_ref = pd.read_csv(os.path.join(RAW, "drug_reference.csv"))
classed = df.merge(drug_ref, on="drug_name", how="left", validate="many_to_one")
classed["drug_class"] = classed["drug_class"].fillna("Unclassified")
dates = pd.to_datetime(classed["event_date"], format="%Y%m%d")
classed["quarter"] = dates.dt.to_period("Q").astype(str)
 
counts = pd.crosstab(classed["quarter"], classed["drug_class"])
share = counts.div(counts.sum(axis=1), axis=0) * 100
print("quarters:", share.shape[0], "| classes:", share.shape[1],
      "| each row adds up to 100:", bool((share.sum(axis=1).round(6) == 100).all()))
 
yoy = share.diff(periods=4)
latest = yoy.loc["2026Q2"].drop("Unclassified")
big = counts.sum() >= 50
latest = latest[big.reindex(latest.index)]
print("named classes with 50+ reports:", len(latest))
 
table = pd.DataFrame({
    "2025Q2_%": share.loc["2025Q2", latest.index],
    "2026Q2_%": share.loc["2026Q2", latest.index],
    "change_pts": latest,
}).round(1)
pd.set_option("display.max_colwidth", 45)   # long class names are cut at 45
print()
print("biggest gains in share points, 2025 Q2 -> 2026 Q2:")
print(table.sort_values("change_pts", ascending=False).head(3))
print()
print("biggest falls:")
print(table.sort_values("change_pts").head(3))

quarters: 8 | classes: 226 | each row adds up to 100: True
named classes with 50+ reports: 20

biggest gains in share points, 2025 Q2 -> 2026 Q2:
                                               2025Q2_%  2026Q2_%  change_pts
drug_class                                                                   
Interleukin-4 Receptor alpha Antagonist             6.7      20.8        14.1
Glucose-dependent Insulinotropic Polypepti...       1.2       4.6         3.5
PCSK9 Inhibitor                                     1.2       2.2         1.0

biggest falls:
                    2025Q2_%  2026Q2_%  change_pts
drug_class                                        
Corticosteroid           1.8       0.2        -1.6
Insulin Analog           2.5       1.0        -1.5
Thalidomide Analog       3.9       2.8        -1.1
